### 1. Data Loading and Sample Selection
First, we import the necessary libraries, load our dataset, and filter for our desired analysis sample (examined participants only).

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("nhanes_cleaned.csv")

EXAMINED_ONLY = True

if EXAMINED_ONLY:
    df = df[df["Interview/Examination status"] == 2].copy()
    weight_col = "Full sample 2-year MEC exam weight"
else:
    df = df.copy()
    weight_col = "Full sample 2-year interview weight"
print(f"Participants: {len(df):,}  (survey weight: {weight_col})")

Participants: 2,794  (survey weight: Full sample 2-year MEC exam weight)


Mobile Examination Center (MEC)

### 2. Recoding Categorical Variables
Next, we map binary responses and categorical demographic strings to more model-friendly numeric representations or simplified labels.

In [2]:
yes_no = {1: 1, 2: 0}

df["female"]               = df["Gender"].map({1: 0, 2: 1})
df["us_born"]              = df["Country of birth"].map({1: 1, 2: 0})
df["veteran"]              = df["Served active duty in US Armed Forces"].map(yes_no)
df["main_meal_planner"]    = df["Main meal planner/preparer"].map(yes_no)
df["shares_meal_planning"] = df["Shared meal planning/preparing duty"].map(yes_no)
df["main_food_shopper"]    = df["Main food shopper"].map(yes_no)
df["shares_food_shopping"] = df["Shared food shopping duty"].map(yes_no)

df["race_eth"] = df["Race/Hispanic origin w/ NH Asian"].map({
    1: "Mexican American", 2: "Other Hispanic", 3: "NH White",
    4: "NH Black", 6: "NH Asian", 7: "Other/Multiracial"})
df["marital"] = df["Marital status"].map({
    1: "Married/partner", 2: "Widowed/divorced/separated", 3: "Never married"})

df["education"] = df["Education level - Adults 20+"]

df = df.rename(columns={
    "Age in years at screening": "age",
    "Ratio of family income to poverty": "income_poverty_ratio",
    "Total number of people in the Household": "household_size",
})

### 3. Feature Selection
Here we establish which columns will be used as the target outcome, behavioral features, covariates, and survey design metadata.

In [3]:
outcome = "pct_wt_change"

behavioral = ["mod_min_wk", "vig_min_wk", "sedentary_hr_day", "phq9_score",
              "drinks_per_week", "smoking_status", "sleep_hr_avg",
              "main_meal_planner", "shares_meal_planning",
              "main_food_shopper", "shares_food_shopping"]

covariates = ["age", "female", "race_eth", "education", "income_poverty_ratio",
              "marital", "household_size", "us_born", "veteran", "bmi_1yr_ago"]

design = ["Respondent sequence number", weight_col,
          "Masked variance pseudo-stratum", "Masked variance pseudo-PSU"]

model_df = df[design + [outcome, "lost_5pct"] + covariates + behavioral].copy()

Behavioral predictors (the focus): activity, sitting time, depression score, drinking, smoking, sleep, food-shopping and meal-planning roles.

Covariates (the background): age, sex, race/ethnicity, education, income, marital status, household size, birthplace, veteran status, starting BMI.

Survey design variables: describe how the person was selected for NHANES, left out of X, not predictors.

Respondent sequence number (SEQN): Unique ID for each person.

MEC exam weight (weight_col): How many people in the U.S. population each participant represents. (weight_col is the number of people in the U.S. population that person represents.)

Masked variance pseudo-stratum: The stratum the person was sampled from.

Masked variance pseudo-PSU: The cluster (county group) within the stratum. (Primary Sampling Unit: PSU, which is counties.)

### 4. One-Hot Encoding
We turn categorical text strings into dummy indicator variables, taking care to restore `NaN` values where the original category was missing.

In [4]:
model_df["smoking_status"] = pd.Categorical(model_df["smoking_status"],
                                            categories=["Never", "Former", "Current"])
model_df["race_eth"] = pd.Categorical(model_df["race_eth"],
    categories=["NH White", "Mexican American", "Other Hispanic", "NH Black",
                "NH Asian", "Other/Multiracial"])
model_df["marital"] = pd.Categorical(model_df["marital"],
    categories=["Married/partner", "Widowed/divorced/separated", "Never married"])

model_df = pd.get_dummies(model_df, columns=["smoking_status", "race_eth", "marital"],
                          drop_first=True, dtype=float, dummy_na=False)

for prefix, src in [("smoking_status", "smoking_status"), ("race_eth", "race_eth"),
                    ("marital", "marital")]:
    missing_rows = df[src].isna()
    dummy_cols = [c for c in model_df.columns if c.startswith(prefix + "_")]
    model_df.loc[missing_rows.values, dummy_cols] = np.nan

### 5. Check Missing Values and Save
We review the percentage of missing values remaining and save the fully prepared dataset for modeling.

In [5]:
miss = (model_df.isna().mean() * 100).round(1)
print("\nColumns with missing values (%):")
print(miss[miss > 0].sort_values(ascending=False).to_string())
print(f"\nComplete cases: {model_df.dropna().shape[0]:,} of {len(model_df):,}")

model_df.to_csv("nhanes_model_ready.csv", index=False)
print(f"\nSaved nhanes_model_ready.csv with shape {model_df.shape}")


Columns with missing values (%):
drinks_per_week           12.4
phq9_score                11.9
income_poverty_ratio      11.6
sleep_hr_avg               1.2
vig_min_wk                 0.5
mod_min_wk                 0.5
bmi_1yr_ago                0.4
sedentary_hr_day           0.3
smoking_status_Former      0.1
smoking_status_Current     0.1

Complete cases: 2,117 of 2,794

Saved nhanes_model_ready.csv with shape (2794, 33)


### 6. Final Model Inputs
Finally, we separate out the features (X), outcome (y), and weights (w) to be ready for machine learning pipelines.

In [6]:
not_features = design + [outcome, "lost_5pct"]
X = model_df.drop(columns=not_features)
y = model_df[outcome]
w = model_df[weight_col]
print(f"\nX: {X.shape}  y: {y.shape}")
print("Features:", list(X.columns))


X: (2794, 27)  y: (2794,)
Features: ['age', 'female', 'education', 'income_poverty_ratio', 'household_size', 'us_born', 'veteran', 'bmi_1yr_ago', 'mod_min_wk', 'vig_min_wk', 'sedentary_hr_day', 'phq9_score', 'drinks_per_week', 'sleep_hr_avg', 'main_meal_planner', 'shares_meal_planning', 'main_food_shopper', 'shares_food_shopping', 'smoking_status_Former', 'smoking_status_Current', 'race_eth_Mexican American', 'race_eth_Other Hispanic', 'race_eth_NH Black', 'race_eth_NH Asian', 'race_eth_Other/Multiracial', 'marital_Widowed/divorced/separated', 'marital_Never married']
